# Autoregressive (자동회귀) 문장 생성

간단한 자동 회귀적인 텍스트 생성을 위한 코드 예제입니다. 여기서는 OpenAI의 GPT-2 모델을 사용하는데, GPT-3은 API를 통해서만 직접 사용할 수 있기 때문입니다. 그럼에도 GPT-2는 GPT-3와 매우 유사한 구조를 가지고 있으므로, 동일한 접근 방식을 사용합니다.

### Colab에서 실행

GPT2LMHeadModel: GPT-2 모델의 구현체 중 하나로, 'LMHead'는 'Language Modeling Head'를 의미합니다. 이 모델은 텍스트를 생성하는 데 사용되며 'Head'는 신경망의 최종 계층을 의미하는데, 이 경우 언어 모델링 작업을 위한 계층입니다.

GPT2Tokenizer: GPT-2 모델에 텍스트를 입력하기 전에, 이 토크나이저를 사용하여 텍스트를 토큰(token)으로 분해하고, 이 토큰들을 모델이 이해할 수 있는 숫자 ID로 변환합니다. 또한, 이 ID들을 모델의 입력 형식에 맞게 패딩(padding)하거나 잘라내기(truncation)하는 작업도 수행합니다.

In [1]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

In [2]:
# GPT2Tokenizer.from_pretrained 함수를 사용해 사전 훈련된 "gpt2-medium" 모델의 토크나이저를 불러옵니다.
tokenizer = GPT2Tokenizer.from_pretrained("gpt2-medium")

# GPT2LMHeadModel.from_pretrained 함수를 사용해 사전 훈련된 "gpt2-medium" 모델을 불러옵니다.
model = GPT2LMHeadModel.from_pretrained("gpt2-medium")

# "Once upon a time"이라는 문자열을 시작으로 하는 텍스트를 준비합니다.
input_text = "Once upon a time"

# tokenizer.encode 함수를 사용해 입력 텍스트를 토큰 ID로 변환합니다.
# return_tensors="pt" 옵션을 사용해 결과를 파이토치 텐서 형태로 받습니다.
input_ids = tokenizer.encode(input_text, return_tensors="pt")

# 변환된 토큰 ID를 출력합니다.
input_ids

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tensor([[7454, 2402,  257,  640]])

In [3]:
# model.generate 함수를 사용해 문장을 생성합니다.
# input_ids: 생성의 시작점이 될 토큰 ID
# max_length: 생성할 문장의 최대 길이를 지정
# num_return_sequences: 반환할 문장 시퀀스의 수를 지정
# pad_token_id: 패딩에 사용될 토큰 ID를 지정 (여기서는 문장의 끝을 나타내는 토큰 ID를 사용)
output = model.generate(input_ids, max_length=50, num_return_sequences=1, pad_token_id=tokenizer.eos_token_id)

# 생성된 문장의 토큰 ID 출력
output

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


tensor([[ 7454,  2402,   257,   640,    11,   612,   373,   257,   582,   508,
          5615,   287,   257,  7404,  1444,   509, 17716,   322,    13,   679,
           373,   257,   845,   922,   582,    11,   290,   339,   373,   845,
          1611,   284,   465,  1751,    13,  1881,  1110,    11,   339,   373,
          6155,  1863,   262,  2975,    11,   290,   339,  2497,   257,  2415]])

In [4]:
# enumerate 함수를 사용해 생성된 문장들을 반복하면서 출력합니다.
for i, generated_text in enumerate(output):
    # tokenizer.decode 함수를 사용해 토큰 ID를 문자열로 디코딩합니다.
    # skip_special_tokens=True 옵션을 주어 특수 토큰들은 건너뛰고 텍스트만 반환하도록 합니다.
    decoded_text = tokenizer.decode(generated_text, skip_special_tokens=True)

    # f-string을 사용해 생성된 텍스트와 함께 순번을 출력합니다.
    print(f"Generated text {i + 1}: {decoded_text}")

Generated text 1: Once upon a time, there was a man who lived in a village called Krakow. He was a very good man, and he was very kind to his children. One day, he was walking along the road, and he saw a woman


GPT-2는 자체적으로 autoregressive 모델입니다. "Autoregressive"란, 이전에 생성된 토큰들을 기반으로 다음 토큰을 생성하는 모델을 의미합니다.

위의 코드에서 `model.generate` 메서드는 이미 autoregressive한 방식으로 문장을 생성합니다. 그러나 이를 명시적으로 보여주기 위해 각 단계에서 토큰을 하나씩 생성하는 autoregressive한 코드를 아래에 작성하겠습니다:

In [5]:
# 문장 시작 부분
input_text = "Once upon a time"
input_ids = tokenizer.encode(input_text, return_tensors="pt")
input_ids

tensor([[7454, 2402,  257,  640]])

input_ids를 입력으로 받아 next token의 확률 분포를 예측값으로 반환 받습니다.

In [6]:
# model 함수에 input_ids를 입력으로 넣어 모델의 예측 결과를 받습니다.
predictions = model(input_ids)

# 예측 결과 중에서 로짓(logits)을 추출합니다.
# 로짓은 모델이 각 토큰에 대해 예측한 점수로, 이 점수는 다음 토큰의 확률 분포로 변환되기 전의 값입니다.
logits = predictions.logits

# 로짓 값을 출력합니다. (배치 크기, 시퀀스 길이, 어휘 사전의 크기)
print(logits.shape)
logits

torch.Size([1, 4, 50257])


tensor([[[ -74.6887,  -74.2220,  -80.2354,  ...,  -83.7431,  -80.4567,
           -76.2926],
         [-225.0849, -225.3535, -229.2500,  ..., -236.9879, -237.3180,
          -224.3200],
         [  -6.1224,   -4.9656,  -10.4817,  ...,  -13.5870,  -12.2557,
            -5.6444],
         [  -9.2670,   -7.4639,  -14.5420,  ...,  -16.8137,  -16.4736,
            -9.4271]]], grad_fn=<UnsafeViewBackward0>)

예측값 중 가장 큰 값의 token을 선택합니다.

In [7]:
# torch.argmax 함수를 사용하여 가장 높은 로짓 값을 가진 토큰의 인덱스를 추출합니다.
# 이 인덱스는 다음에 올 토큰의 ID를 나타냅니다.
# logits[0, -1]는 마지막 단어에 대한 로짓 값을 의미합니다.
predicted_token_id = torch.argmax(logits[0, -1]).item()

# 예측된 토큰 ID를 출력합니다.
print(f"예측한 next token ID: {predicted_token_id}")

# tokenizer.decode 함수를 사용하여 토큰 ID를 실제 텍스트로 디코딩합니다.
# skip_special_tokens=True 옵션을 주어 특수 토큰을 제외한 순수 텍스트만 반환하도록 합니다.
token = tokenizer.decode(predicted_token_id, skip_special_tokens=True)

# 디코딩된 텍스트를 출력합니다.
print(f"decoding한 next token : {token}")

예측한 next token ID: 11
decoding한 next token : ,


위와 같은 방식으로 마지막 단어 이어 붙이기 형식의 문장 생성을 합니다.

In [8]:
# Autoregressive한 방식으로 문장 생성
max_length = 20                     # 생성할 문장의 최대 길이 설정
input_ids_concat = input_ids  # 초기 입력 토큰(예: 문장의 시작 부분)

# 생성할 문장의 길이가 max_length보다 작을 동안 반복
while input_ids_concat.shape[1] < max_length:
    # 모델을 사용하여 다음 토큰 예측
    predictions = model(input_ids_concat)
    # 예측된 로짓을 가져옴
    logits = predictions.logits
    # 로짓에서 가장 확률이 높은 토큰을 선택
    predicted_token = torch.argmax(logits[0, -1]).item()

    # 예측된 토큰을 기존의 입력 토큰 뒤에 추가
    input_ids_concat = torch.cat([input_ids_concat, torch.tensor([[predicted_token]])], dim=1)
    # 현재까지 예측된 토큰들을 출력
    print(input_ids_concat)

# 디코딩하여 최종적으로 생성된 Text 출력
decoded_text = tokenizer.decode(input_ids_concat[0], skip_special_tokens=True)
print(decoded_text)

tensor([[7454, 2402,  257,  640,   11]])
tensor([[7454, 2402,  257,  640,   11,  612]])
tensor([[7454, 2402,  257,  640,   11,  612,  373]])
tensor([[7454, 2402,  257,  640,   11,  612,  373,  257]])
tensor([[7454, 2402,  257,  640,   11,  612,  373,  257,  582]])
tensor([[7454, 2402,  257,  640,   11,  612,  373,  257,  582,  508]])
tensor([[7454, 2402,  257,  640,   11,  612,  373,  257,  582,  508, 5615]])
tensor([[7454, 2402,  257,  640,   11,  612,  373,  257,  582,  508, 5615,  287]])
tensor([[7454, 2402,  257,  640,   11,  612,  373,  257,  582,  508, 5615,  287,
          257]])
tensor([[7454, 2402,  257,  640,   11,  612,  373,  257,  582,  508, 5615,  287,
          257, 7404]])
tensor([[7454, 2402,  257,  640,   11,  612,  373,  257,  582,  508, 5615,  287,
          257, 7404, 1444]])
tensor([[7454, 2402,  257,  640,   11,  612,  373,  257,  582,  508, 5615,  287,
          257, 7404, 1444,  509]])
tensor([[ 7454,  2402,   257,   640,    11,   612,   373,   257,   582,   50